In [12]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# --- robust repo root ---
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.reverse_vrp import (
    build_distance_matrix,
    build_reverse_vrp_nodes,
    solve_reverse_cvrptw,
    parse_solution,
)

DATA_DIR   = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

VEHICLE_CAPACITY_G      = 500_000
VEHICLE_SPEED_KMH       = 40
FIXED_COST_PER_ROUTE    = 50
VAR_COST_PER_KM         = 1.5
SERVICE_TIME_MIN        = 5
SOLVER_TIME_LIMIT_S     = 30
MAX_CUSTOMERS_PER_ZONE  = 75
RETURN_PROB_THRESHOLD   = 0.30

print('Setup complete. Repo root:', REPO_ROOT)


Setup complete. Repo root: /Users/pranavt/Downloads/PGDBA/IIT KGP/SCA/project/SCA_DARK_STORES


In [13]:
# Load v3 — has return_prob and return_flag from classifier
master_df   = pd.read_parquet(DATA_DIR / "master_df_v3.parquet")
dark_stores = pd.read_csv(DATA_DIR / "dark_stores_final.csv")

# Filter to flagged return orders only
return_df = master_df[master_df["return_flag"] == 1].copy()

print(f"Total orders : {len(master_df):,}")
print(f"Return nodes : {len(return_df):,} ({len(return_df)/len(master_df)*100:.1f}%)")
print(f"Dark stores  : {len(dark_stores)}")


Using real return_prob from classifier.
Total orders     : 41,731
Return nodes     : 466 (1.1%)
Dark stores      : 11


In [14]:
reverse_zones = build_reverse_vrp_nodes(
    return_df, dark_stores,
    max_per_zone=MAX_CUSTOMERS_PER_ZONE,
)


Built reverse VRP nodes for 11 zones
  Zone  0: 25 pickups | total weight = 27.7 kg
  Zone  1: 48 pickups | total weight = 145.5 kg
  Zone  2: 31 pickups | total weight = 64.2 kg
  Zone  3: 70 pickups | total weight = 161.6 kg
  Zone  4: 19 pickups | total weight = 32.6 kg
  Zone  5: 33 pickups | total weight = 78.7 kg
  Zone  6: 75 pickups | total weight = 103.9 kg
  Zone  7: 25 pickups | total weight = 97.8 kg
  Zone  8: 41 pickups | total weight = 42.9 kg
  Zone  9: 28 pickups | total weight = 49.5 kg
  Zone 10: 36 pickups | total weight = 94.1 kg


In [15]:
rows = []
for z in reverse_zones:
    for i, (coords, demand, tw, nid) in enumerate(zip(
            z["node_coords"], z["demands"], z["time_windows"], z["node_ids"])):
        rows.append({
            "zone_id":  z["zone_id"],
            "node_idx": i,
            "node_id":  nid,
            "lat":      round(float(coords[0]), 6),
            "lon":      round(float(coords[1]), 6),
            "demand_g": int(demand),
            "tw_open":  tw[0],
            "tw_close": tw[1],
            "is_depot": int(i == 0),
        })

reverse_vrp_nodes = pd.DataFrame(rows)
reverse_vrp_nodes.to_csv(DATA_DIR / "reverse_vrp_nodes.csv", index=False)
print(f"Saved: {DATA_DIR / 'reverse_vrp_nodes.csv'}  ({len(reverse_vrp_nodes)} rows)")
reverse_vrp_nodes.head(6)

Saved: /Users/pranavt/Downloads/PGDBA/IIT KGP/SCA/project/SCA_DARK_STORES/data/reverse_vrp_nodes.csv  (442 rows)


,zone_id,node_idx,node_id,lat,lon,demand_g,tw_open,tw_close,is_depot
0,0,0,depot,-23.563536,-46.543798,0,0,1440,1
1,0,1,1719,-23.542273,-46.568867,833,480,720,0
2,0,2,1849,-23.584691,-46.561372,3050,720,1080,0
3,0,3,2243,-23.559719,-46.571297,1700,480,720,0
4,0,4,4589,-23.543947,-46.517405,1350,720,1080,0
5,0,5,5315,-23.580113,-46.498297,167,480,720,0


In [16]:
def solve_reverse_cvrptw(zone: dict, num_vehicles: int = 10) -> dict:
    n       = len(zone["node_coords"])
    demands = zone["demands"].tolist()
    tw      = zone["time_windows"]

    dist_matrix  = build_distance_matrix(zone["node_coords"])
    speed_m_per_min = VEHICLE_SPEED_KMH * 1000 / 60
    time_matrix  = np.rint(dist_matrix / speed_m_per_min).astype(int)

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def dist_cb(i, j):
        return int(dist_matrix[manager.IndexToNode(i)][manager.IndexToNode(j)])
    dist_cb_idx = routing.RegisterTransitCallback(dist_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(dist_cb_idx)

    def demand_cb(i):
        return int(demands[manager.IndexToNode(i)])
    dem_cb_idx = routing.RegisterUnaryTransitCallback(demand_cb)
    routing.AddDimensionWithVehicleCapacity(
        dem_cb_idx, 0,
        [VEHICLE_CAPACITY_G] * num_vehicles,
        True, "Capacity")

    def time_cb(i, j):
        node_i  = manager.IndexToNode(i)
        travel  = int(time_matrix[node_i][manager.IndexToNode(j)])
        service = SERVICE_TIME_MIN if node_i != 0 else 0
        return travel + service
    time_cb_idx = routing.RegisterTransitCallback(time_cb)
    routing.AddDimension(time_cb_idx, 60, 1440, False, "Time")
    time_dim = routing.GetDimensionOrDie("Time")
    for node_idx, (open_t, close_t) in enumerate(tw):
        time_dim.CumulVar(manager.NodeToIndex(node_idx)).SetRange(open_t, close_t)

    penalty = 100_000
    for node in range(1, n):
        routing.AddDisjunction([manager.NodeToIndex(node)], penalty)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
    params.time_limit.seconds = SOLVER_TIME_LIMIT_S

    assignment = routing.SolveWithParameters(params)

    if assignment is None:
        print(f"  Zone {zone['zone_id']}: NO SOLUTION FOUND")
        return {"zone_id": zone["zone_id"], "solved": False}

    routes_df, summary = parse_solution(
        manager, routing, assignment,
        zone["node_coords"], zone["node_ids"],
        distance_matrix=dist_matrix
    )
    routes_df["zone_id"] = zone["zone_id"]

    total_dist_km = summary["total_distance_km"]
    n_vehicles    = summary["n_vehicles_used"]
    routing_cost  = FIXED_COST_PER_ROUTE * n_vehicles + VAR_COST_PER_KM * total_dist_km

    print(f"  Zone {zone['zone_id']:2d}: {n_vehicles} vehicles | "
          f"{total_dist_km:.1f} km | R${routing_cost:.0f}")

    return {
        "zone_id":         zone["zone_id"],
        "solved":          True,
        "routes_df":       routes_df,
        "summary":         summary,
        "total_dist_km":   total_dist_km,
        "n_vehicles":      n_vehicles,
        "routing_cost_R$": routing_cost,
        "dist_matrix":     dist_matrix,
    }

print("Reverse solver function defined.")

Reverse solver function defined.


In [17]:
print("Running Reverse VRP (CVRPTW) — pickup routes...")
print(f"Time limit: {SOLVER_TIME_LIMIT_S}s per zone | "
      f"Vehicle capacity: {VEHICLE_CAPACITY_G/1000:.0f} kg\n")

reverse_results = {}
for z in reverse_zones:
    result = solve_reverse_cvrptw(z)
    reverse_results[z["zone_id"]] = result

n_solved = sum(r["solved"] for r in reverse_results.values())
print(f"\nCompleted: {n_solved}/{len(reverse_zones)} zones solved.")

Running Reverse VRP (CVRPTW) — pickup routes...
Time limit: 30s per zone | Vehicle capacity: 500 kg

  Zone  0: 2 vehicles | 60.0 km | R$190
  Zone  1: 3 vehicles | 62.1 km | R$243
  Zone  2: 3 vehicles | 462.4 km | R$844
  Zone  3: 4 vehicles | 1137.7 km | R$1907
  Zone  4: 2 vehicles | 190.3 km | R$386
  Zone  5: 3 vehicles | 413.8 km | R$771
  Zone  6: 4 vehicles | 1100.4 km | R$1851
  Zone  7: 2 vehicles | 67.7 km | R$202
  Zone  8: 2 vehicles | 210.5 km | R$416
  Zone  9: 2 vehicles | 66.5 km | R$200
  Zone 10: 4 vehicles | 72.9 km | R$309

Completed: 11/11 zones solved.


In [18]:
all_route_dfs = [r["routes_df"] for r in reverse_results.values()
                 if r["solved"] and "routes_df" in r]
reverse_routes_df = pd.concat(all_route_dfs, ignore_index=True)
reverse_routes_df.to_csv(OUTPUT_DIR / "reverse_routes.csv", index=False)

json_out = []
for zid, r in reverse_results.items():
    if not r["solved"]: continue
    json_out.append({
        "zone_id":         zid,
        "n_vehicles":      r["n_vehicles"],
        "total_dist_km":   r["total_dist_km"],
        "routing_cost_R$": r["routing_cost_R$"],
        "routes":          r["routes_df"].to_dict(orient="records"),
    })
with open(OUTPUT_DIR / "reverse_routes.json", "w") as f:
    json.dump(json_out, f, indent=2)

kpi_rows = []
for zid, r in reverse_results.items():
    if not r["solved"]: continue
    kpi_rows.append({
        "zone_id":         zid,
        "n_pickups":       next(z["n_pickups"] for z in reverse_zones if z["zone_id"] == zid),
        "n_vehicles_used": r["n_vehicles"],
        "total_dist_km":   round(r["total_dist_km"], 2),
        "routing_cost_R$": round(r["routing_cost_R$"], 2),
        "max_route_km":    r["summary"]["max_route_km"],
        "min_route_km":    r["summary"]["min_route_km"],
    })
kpi_df = pd.DataFrame(kpi_rows)
kpi_df.to_csv(OUTPUT_DIR / "reverse_kpi_summary.csv", index=False)

print("Saved:")
print(f"  {OUTPUT_DIR / 'reverse_routes.csv'}")
print(f"  {OUTPUT_DIR / 'reverse_routes.json'}")
print(f"  {OUTPUT_DIR / 'reverse_kpi_summary.csv'}")
print()
print(kpi_df.to_string(index=False))

Saved:
  /Users/pranavt/Downloads/PGDBA/IIT KGP/SCA/project/SCA_DARK_STORES/outputs/reverse_routes.csv
  /Users/pranavt/Downloads/PGDBA/IIT KGP/SCA/project/SCA_DARK_STORES/outputs/reverse_routes.json
  /Users/pranavt/Downloads/PGDBA/IIT KGP/SCA/project/SCA_DARK_STORES/outputs/reverse_kpi_summary.csv

 zone_id  n_pickups  n_vehicles_used  total_dist_km  routing_cost_R$  max_route_km  min_route_km
       0         25                2          59.98           189.97         34.60         25.38
       1         48                3          62.06           243.09         27.86         16.11
       2         31                3         462.39           843.58        386.04         27.16
       3         70                4        1137.74          1906.61        654.31         22.33
       4         19                2         190.34           385.51        171.29         19.05
       5         33                3         413.83           770.74        255.84         14.98
       6         75

In [20]:
fwd_kpi = pd.read_csv(OUTPUT_DIR / "forward_kpi_summary.csv")
rev_kpi = kpi_df.copy()

merged = fwd_kpi.merge(rev_kpi, on="zone_id", suffixes=("_fwd", "_rev"))

print("=" * 60)
print("FORWARD vs REVERSE VRP — ZONE-LEVEL COMPARISON")
print("=" * 60)
print(merged[["zone_id",
              "n_customers", "n_pickups",
              "total_dist_km_fwd", "total_dist_km_rev",
              "routing_cost_R$_fwd", "routing_cost_R$_rev"]].to_string(index=False))

print("\n" + "=" * 60)
print("TOTALS")
print("=" * 60)
print(f"  Forward  — dist: {fwd_kpi['total_dist_km'].sum():.1f} km | "
      f"cost: R${fwd_kpi['routing_cost_R$'].sum():.0f} | "
      f"vehicles: {fwd_kpi['n_vehicles_used'].sum()}")
print(f"  Reverse  — dist: {rev_kpi['total_dist_km'].sum():.1f} km | "
      f"cost: R${rev_kpi['routing_cost_R$'].sum():.0f} | "
      f"vehicles: {rev_kpi['n_vehicles_used'].sum()}")
total_cost = fwd_kpi['routing_cost_R$'].sum() + rev_kpi['routing_cost_R$'].sum()
print(f"\n  Combined total cost: R${total_cost:.0f}")
print("=" * 60)

FORWARD vs REVERSE VRP — ZONE-LEVEL COMPARISON
 zone_id  n_customers  n_pickups  total_dist_km_fwd  total_dist_km_rev  routing_cost_R$_fwd  routing_cost_R$_rev
       0           75         25             112.72              59.98               269.08               189.97
       1           75         48              84.52              62.06               276.78               243.09
       2           75         31             691.08             462.39              1236.62               843.58
       3           75         70             678.49            1137.74              1167.74              1906.61
       4           75         19             681.08             190.34              1171.62               385.51
       5           75         33             966.72             413.83              1650.08               770.74
       6           75         75            1169.09            1100.37              1953.63              1850.55
       7           75         25              90.